In [167]:
from transformers import AutoModel, AutoTokenizer
import torch
from IPython.display import display, HTML

model = AutoModel.from_pretrained("meta-llama/Llama-3.2-3B", device_map="auto")
model.eval()
model.set_attn_implementation('eager')
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [169]:
def convert_ids_to_toklist(ids: torch.Tensor):
    tokens = tokenizer.convert_ids_to_tokens(ids)
    return [tokenizer.convert_tokens_to_string([tok]) for tok in tokens]

def float2alpha(w: float):
    assert w <= 1
    value = int(255*w)
    hex_code = hex(value)[2:]
    return hex_code if len(hex_code) == 2 else "0" + hex_code

def colorize(words, color_array):
    # words is a list of words
    # color_array is an array of numbers between 0 and 1 of length equal to words
    color_normalized = (color_array / max(color_array))**(1/2)
    template = '<span style="color: black; background-color: #5d54ff{}">{}</span>'
    colored_string = '<div style="background-color: white">'
    for word, color in zip(words, color_normalized):
        alpha = float2alpha(color)
        colored_string += template.format(alpha, ' ' + word + ' ')
    return colored_string + "</div>"

@torch.no_grad()
def vizualize_attention(prompt: str):
    tokens = tokenizer([prompt], add_special_tokens=True, return_tensors="pt")
    input_ids = tokens.input_ids.cuda()
    att_mask = tokens.attention_mask.cuda()
    outputs = model(input_ids, att_mask, output_attentions=True)
    att_tensor = torch.stack(outputs.attentions)
    attentions = att_tensor.squeeze().mean((0,1))[-1].cpu()
    toklist = convert_ids_to_toklist(input_ids.squeeze())

    s = colorize(toklist[1:], attentions[1:])
    display(HTML(s))

In [172]:
prompt = """> souhrnné označení veškeré hmoty, energie a časoprostoru
$ vesmír

> třetí planeta od Slunce ve Sluneční soustavě
$ Země

> schopnost získávat energii ze svého okolí pro účely rozmnožování
$ život

> stav organismu po ukončení života
$ smrt

> člen lidské společnosti, jediný žijící druh rodu Homo
$ člověk

> Politický systém je založen na svobodném a dobrovolném vzniku a volné soutěži politických stran respektujících základní demokratické principy a odmítajících násilí jako prostředek k prosazování svých zájmů.
$"""

vizualize_attention(prompt)

In [174]:
prompt = """Politický systém je založen na svobodném a dobrovolném vzniku a volné soutěži politických stran respektujících základní demokratické principy a odmítajících násilí jako prostředek k prosazování svých zájmů."""
vizualize_attention(prompt)